In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import warnings
from typing import Literal

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot
from pandas.plotting import autocorrelation_plot
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from utility import (
    calculate_data_masks,
    calculate_residuals,
    plot_fouling_summary,
    plot_timeseries_grid,
    predict_fouling_onset,
    prepare_model_input,
    train_model,
    visualize_correlations,
    visualize_data_folds,
    visualize_imputation,
    visualize_learning_curve,
)

warnings.filterwarnings("ignore")


In [3]:
RANDOM_SEED = 14
rng = np.random.default_rng(RANDOM_SEED)

In [4]:
image_folder = "plots"
os.makedirs(image_folder, exist_ok=True)

In [5]:
data_raw = pd.read_csv("ds_compressor_data.csv")
data_raw["timestamp"] = pd.to_datetime(data_raw["timestamp"], utc=True)
data_raw = data_raw.set_index("timestamp").sort_index()

Compressor data spans 8 months from August 2024 through March 2025 daily (timestamps are evenly spaced at 1-day intervals).

In [6]:
data_raw.index.diff().unique()  # type: ignore

TimedeltaIndex([NaT, '1 days'], dtype='timedelta64[ns]', name='timestamp', freq=None)

In [7]:
data_raw.index.min()

Timestamp('2024-08-01 00:00:00+0000', tz='UTC')

In [8]:
data_raw.index.max()

Timestamp('2025-03-30 00:00:00+0000', tz='UTC')

In [9]:
data_raw.columns

Index(['Inlet_Temperature', 'Ambient_Humidity', 'Off_Gas_Fraction_Percentage',
       'Outlet_Flow_Rate_SP', 'Outlet_Pressure_SP', 'Outlet_Temperature',
       'Inlet_Flow_Rate', 'Inlet_Pressure', 'Outlet_Flow_Rate',
       'Outlet_Pressure'],
      dtype='object')

Off_Gas_Fraction_Percentage: This represents the percentage of off-gas (or residual gas) in the system. Off-gas is the gas that is not fully compressed or utilized and can affect the efficiency and output of the compressor.

Outlet_Flow_Rate_SP: This stands for the Set Point (SP) for the outlet flow rate. A set point is a target value that the system aims to maintain. In this case, it's the desired flow rate of the gas or fluid exiting the compressor.

Outlet_Pressure_SP: Similar to the outlet flow rate SP, this is the set point for the pressure at the compressor's outlet. It's the target pressure that the system strives to achieve.

Outlet_Flow_Rate: This is the actual flow rate of the gas or fluid exiting the compressor. It can be compared to the set point (Outlet_Flow_Rate_SP) to evaluate the compressor's performance.

Outlet_Pressure: This column records the actual pressure of the gas or fluid at the compressor's outlet. It can be compared to the set point (Outlet_Pressure_SP) to assess how well the compressor is maintaining the desired pressure.


In [10]:
plot_config = [
    [("Off_Gas_Fraction_Percentage", "b")],
    [("Ambient_Humidity", "y")],
    [("Outlet_Temperature", "g"), ("Inlet_Temperature", "c")],
    [("Inlet_Flow_Rate", "y"), ("Outlet_Flow_Rate", "b"), ("Outlet_Flow_Rate_SP", "r")],
    [("Inlet_Pressure", "y"), ("Outlet_Pressure", "b"), ("Outlet_Pressure_SP", "r")],
]
fname = os.path.join(image_folder, "data_time_series.png")
plot_timeseries_grid(data_raw, plot_config, fname)

The core challenge, as stated, is that "outlet pressure is influenced by multiple operating parameters." Therefore, we cannot simply set a hard threshold (e.g., "Alert if pressure < 50 psi") because the pressure changes based on the set point (SP) and inlet conditions.

### Strategy: "Expected vs. Actual" (Residual Analysis)

Since we need to isolate the fouling effect from normal operating changes, the best approach is to build a Virtual Sensor (Regression Model) which solves the specific "Isolating Effects" problem by including Setpoints, Flow, Inlet Pressure, Temperature, ... as inputs. The model "knows" that pressure should drop if flow drops. It won't flag that as a fouling status. It will only flag when pressure drops unaccountably. We train a model to predict what the Outlet Pressure should be given the current inlet conditions and set points. In other words, we need a baseline estimator. Since the machine is controlled, Outlet Pressure ~ Setpoint is over 90% of the answer. We need a linear regression to adjust that baseline slightly based on other operating variables:
- $\text{Output} \approx \text{Setpoint} + \text{Efficiency\_Correction}$

Hypothesis: If the machine is clean, Actual Pressure ≈ Predicted Pressure.
Fouling Signal: If the machine is fouling, Actual Pressure < Predicted Pressure. The gap between them (the residual) is the fouling indicator.


The Solution: Take very important note that we treat this as a Physics Problem, not a Forecasting Problem. We are building a Virtual Sensor, not a Stock Market Predictor.

- Forecasting: Predicting $Y_{t+1}$ based on $Y_{t}$ (Time matters strictly).
- Virtual Sensing: Predicting $Y_{t+1}$ based on $X_t$ (Physics matters).

We use the classical regression model, to only allow the dependent variable to be influenced by current values of the
independent variables (The model is "blind" to the past).
The fact that we broke the autocorrelation link (by removing lags) is exactly what makes this an anomaly detector.

Imagine the Fouling Scenario (January 2025):

1. Inputs (X): Setpoint is still 100. Flow is still 50.
2. The "Physics" Model: Looks at X. Says: "Based on Flow 50 and SP 100, the pressure MUST be 100."
3. The Reality (Y): The machine is fouled. Actual pressure is 80.
4. The Residual: 100 − 80 = +20 (Anomaly Detected!)

Contrast this with the "Autocorrelated/Forecasting" Model:

1. Inputs $Y_{t−1}$: The pressure 1 day ago was 100.
2. The "Lazy" Model: Looks at $Y_{t−1}$ and says: "Since it was 100 a day ago, it's probably 100 now."
3. The Residual: 100 − 100 = 0 (Anomaly Missed!)

Note that we have 4 pressure and flow setpoints (disregard the shutin period). The physics of the compressor $ \text{Pressure} = f\left(\text{Setpoint}, \text{Flow}, \text{Temp}, \dots \right) $ does not change from day to day (data sample intervals) and from setpoint to setpoint. Therefore, we shuffle the data for validation (and to break autocorrelation) as we want to assure model has learned pure physics. This ensures that every training fold contains a mix of all the four setpoints
.

### Summary for the Stakeholder

"We will build a digital twin of the compressor using historical 'healthy' data. This model will predict what the outlet pressure should be based on current operating conditions (flow, temperature, inlet pressure). We will then track the deviation between the model's prediction and the actual sensor reading. A growing deviation indicates fouling, allowing us to alert maintenance before efficiency drops below critical levels."

In [11]:
data_nonzero_setpoints = data_raw[data_raw["Outlet_Flow_Rate"] > 0]
fig, ax = pyplot.subplots(1, 1, figsize=(15, 7))
sns.heatmap(data_nonzero_setpoints.isna(), cbar=False, ax=ax)
display(data_nonzero_setpoints.isna().mean().sort_values(ascending=False))
fname = os.path.join(image_folder, "missing_data_map.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

Outlet_Temperature             0.120773
Inlet_Temperature              0.106280
Off_Gas_Fraction_Percentage    0.091787
Outlet_Pressure                0.028986
Ambient_Humidity               0.024155
Inlet_Pressure                 0.004831
Outlet_Pressure_SP             0.000000
Outlet_Flow_Rate_SP            0.000000
Inlet_Flow_Rate                0.000000
Outlet_Flow_Rate               0.000000
dtype: float64

In [12]:
fig, axs = pyplot.subplots(1, 4, figsize=(20, 10))
visualize_imputation(axs[0], data_raw, feature="Outlet_Temperature")
visualize_imputation(axs[1], data_raw, feature="Inlet_Temperature")
visualize_imputation(axs[2], data_raw, feature="Off_Gas_Fraction_Percentage")
visualize_imputation(axs[3], data_raw, feature="Outlet_Pressure")
fname = os.path.join(image_folder, "missing_data_filling_strategies.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

Avoid: Mean/median imputation - destroys temporal relationships in process data. The time method does a better job by simply interpolating missing values within number of consecutive NaNs to fill. I initially drop rows with missing Outlet_Pressure as the target with limited missing values and replace missing values through simple interpolate method ('time') that preserves distributions (temporal relationships) to a great extent.  

In [13]:
data, X, y = prepare_model_input(data_raw, None, feature_engineering_allowed=True)

In [14]:
baseline_mask, shutin_mask = calculate_data_masks(
    data, rolling_mean_std_multiplier=1.25
)
X_baseline = X.loc[baseline_mask]
y_baseline = y.loc[baseline_mask]

In [15]:
baseline = [("2024-08-01", "2024-11-03"), ("2025-01-01", "2025-02-09")]
fig, ax = pyplot.subplots(1, 1, figsize=(15, 7))
ax.plot(baseline_mask * 1)
for start, end in baseline:
    ax.axvspan(start, end, color="red", alpha=0.3)
fname = os.path.join(image_folder, "baseline.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

### Investigate correlations

In [16]:
data_baseline = data.loc[baseline_mask]

In [17]:
correlation_methods: list[Literal["pearson", "kendall", "spearman"]] = [
    "pearson",
    "kendall",
    "spearman",
]
for corr_method in correlation_methods:
    fig, ax = pyplot.subplots(1, 1, figsize=(15, 7))
    sns.heatmap(data_baseline.corr(method=corr_method), annot=True, fmt=".2f", ax=ax)
    print(corr_method)
    display(
        data_baseline.corr(method=corr_method)["Outlet_Pressure"].sort_values(
            ascending=False, key=abs
        )
    )
    fname = os.path.join(image_folder, f"{corr_method}_corr_hatmap.png")
    pyplot.savefig(fname, dpi=150, bbox_inches="tight")
    pyplot.close(fig)


pearson


Outlet_Pressure                1.000000
Outlet_Pressure_SP             0.997299
Outlet_Flow_Rate_SP            0.996474
Inlet_Pressure                 0.994905
Inlet_Flow_Rate                0.988234
Temp_x_Flow                    0.987086
Inlet_Pressure_x_Flow          0.985573
Outlet_Flow_Rate               0.985225
Temperature_Rise              -0.320304
Inlet_Temperature              0.164146
Off_Gas_Fraction_Percentage   -0.084845
Ambient_Humidity               0.032628
Outlet_Temperature            -0.008911
Name: Outlet_Pressure, dtype: float64

kendall


Outlet_Pressure                1.000000
Outlet_Flow_Rate_SP            0.866731
Outlet_Pressure_SP             0.866731
Inlet_Pressure                 0.768873
Inlet_Pressure_x_Flow          0.733095
Inlet_Flow_Rate                0.731664
Temp_x_Flow                    0.725939
Outlet_Flow_Rate               0.725701
Temperature_Rise              -0.239117
Inlet_Temperature              0.118664
Off_Gas_Fraction_Percentage   -0.035897
Ambient_Humidity               0.027788
Outlet_Temperature             0.007275
Name: Outlet_Pressure, dtype: float64

spearman


Outlet_Pressure                1.000000
Outlet_Flow_Rate_SP            0.966474
Outlet_Pressure_SP             0.966474
Inlet_Pressure                 0.941460
Inlet_Pressure_x_Flow          0.927908
Inlet_Flow_Rate                0.925543
Temp_x_Flow                    0.923506
Outlet_Flow_Rate               0.923375
Temperature_Rise              -0.346669
Inlet_Temperature              0.168843
Off_Gas_Fraction_Percentage   -0.050922
Ambient_Humidity               0.040506
Outlet_Temperature             0.007808
Name: Outlet_Pressure, dtype: float64

In [18]:
scaler = StandardScaler()
data_baseline_scaled = pd.DataFrame(
    scaler.fit_transform(data_baseline),
    columns=data_baseline.columns,
    index=data_baseline.index,
)
target_mask = data_baseline_scaled.columns != "Outlet_Pressure"
mi_scores = mutual_info_regression(
    data_baseline_scaled.loc[:, target_mask],
    data_baseline_scaled["Outlet_Pressure"],
)
mi_series = pd.Series(
    mi_scores, index=data_baseline_scaled.columns[target_mask]
).sort_values(ascending=False)
print("Mutual Information Scores:")
print(mi_series)

Mutual Information Scores:
Inlet_Pressure_x_Flow          1.419672
Inlet_Pressure                 1.393253
Outlet_Flow_Rate_SP            1.388929
Outlet_Pressure_SP             1.388929
Temp_x_Flow                    1.348234
Inlet_Flow_Rate                1.318942
Outlet_Flow_Rate               1.225120
Temperature_Rise               0.251171
Ambient_Humidity               0.019486
Off_Gas_Fraction_Percentage    0.011448
Outlet_Temperature             0.004965
Inlet_Temperature              0.000000
dtype: float64


In [19]:
fig, axs = pyplot.subplots(2, 3, figsize=(20, 15))
visualize_correlations(
    axs[0, 0], data_baseline, "Inlet_Temperature", "Outlet_Temperature"
)
visualize_correlations(axs[0, 1], data_baseline, "Inlet_Flow_Rate", "Outlet_Flow_Rate")
visualize_correlations(axs[0, 2], data_baseline, "Inlet_Flow_Rate", "Inlet_Pressure")
visualize_correlations(
    axs[1, 0], data_baseline, "Outlet_Temperature", "Outlet_Pressure"
)
visualize_correlations(axs[1, 1], data_baseline, "Outlet_Flow_Rate", "Outlet_Pressure")
# non-linear variable
visualize_correlations(
    axs[1, 2], data_baseline, "Inlet_Pressure_x_Flow", "Outlet_Pressure", aspect="auto"
)
fname = os.path.join(image_folder, "correlation_visualizations.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

# Linear Regression

Shuffle aggressively.

- It breaks "Time" so you only learn "Physics."
- It ensures you aren't just memorizing the sequence of events.

In [20]:
n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
cv_splits_stratified = []
stratify_on = X_baseline["Outlet_Pressure_SP"]
for train_idx, test_idx in skf.split(X_baseline, stratify_on):
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    cv_splits_stratified.append((train_idx.copy(), test_idx.copy()))
fname = os.path.join(image_folder, f"{n_splits}_cv_splits_stratified.png")
visualize_data_folds(X_baseline, y_baseline, cv_splits_stratified, fname)

In [21]:
lr_model = train_model(
    X_baseline, y_baseline, cv_splits_stratified, "lr", n_trials=1000
)

[I 2026-03-21 16:06:22,602] A new study created in memory with name: compressor_outlet_pressure
[I 2026-03-21 16:06:24,161] Trial 0 finished with value: -2.6735779119734304 and parameters: {'alpha': 0.0032205410010050263, 'l1_ratio': 0.023614398857099348, 'fit_intercept': True}. Best is trial 0 with value: -2.6735779119734304.
[I 2026-03-21 16:06:25,527] Trial 1 finished with value: -127.42235359315897 and parameters: {'alpha': 0.010749843075795797, 'l1_ratio': 0.17196504219488443, 'fit_intercept': False}. Best is trial 0 with value: -2.6735779119734304.
[I 2026-03-21 16:06:26,884] Trial 2 finished with value: -127.40206504807868 and parameters: {'alpha': 0.4253765302490476, 'l1_ratio': 0.26266041300434534, 'fit_intercept': False}. Best is trial 0 with value: -2.6735779119734304.
[I 2026-03-21 16:06:28,212] Trial 3 finished with value: -127.3480637625966 and parameters: {'alpha': 15.772521651709543, 'l1_ratio': 0.07050317912148582, 'fit_intercept': False}. Best is trial 0 with value: -

Best trial:
  Value: -2.595478749836137
  Params: 
    alpha: 0.08918515076711378
    l1_ratio: 0.9998760812149181
    fit_intercept: True

Baselibe Training MAE: 2.4754
Baseline Training R^2: 0.9947
Baseline Training adjusted R^2: 0.9945
Baseline Residuals Mean: 0.000
Baseline Residuals Std: 2.825


In [22]:
fname = os.path.join(image_folder, "linear_model_learning_curve.png")
visualize_learning_curve(
    lr_model,
    X_baseline,
    y_baseline,
    np.linspace(0.1, 1.0, 10),
    cv_splits_stratified,
    "neg_mean_absolute_error",
    True,
    True,
    RANDOM_SEED,
    fname,
)

Data Sufficiency: 

The lines are flat after ~60 samples (2 months of history). This answers the question: "Do I need more data?" Answer: No. Adding 1,000 more samples won't lower the error. We have fully extracted the signal. The remaining error (~2.5 psi) is just the irreducible noise of the sensors and the process.

In [23]:
coefs = lr_model.named_steps["regressor"].coef_
order = np.argsort(np.abs(coefs))[::-1]
sorted_features = X_baseline.columns[order].tolist()
sorted_coefs = coefs[order]

fig, ax = pyplot.subplots(figsize=(25, 15))
disp = PartialDependenceDisplay.from_estimator(
    lr_model,
    X_baseline,
    sorted_features,
    kind="both",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    ax=ax,
)

# Add coefficient to each subplot
for i, (_, coef) in enumerate(zip(sorted_features, sorted_coefs, strict=False)):
    this_ax = disp.axes_.ravel()[i]
    this_ax.text(
        0.05,
        0.95,
        f"β = {coef:.3f}",
        transform=this_ax.transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.7},
    )


pyplot.tight_layout()
fname = os.path.join(image_folder, "linear_model_partial_dependence.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

1. The Mystery of the "Four Parallel Bands"

    We will notice that in every plot, the blue lines are not scattered randomly. They are grouped into 4 distinct "bands" or clusters stacked vertically. What are these? These are our Operating Regimes (100 psi, 180 psi, 140 psi and 80 psi).

    The Insight: This visual proves that the model separates the "Regime" (the intercept/vertical level) from the "Physics" (the slope).
    Why it’s good: Since the lines within the bands are roughly parallel, it means the physical response is consistent.

    Example (Inlet Temperature): Whether the machine is running at 100 psi (bottom band) or 80 psi (top band), increasing the inlet temperature always increases the outlet pressure by the same relative amount (the slope is identical). This confirms our assumption that the physics is "Time-Invariant" and consistent across regimes.

2. Physics Validation (The Slopes)
    The Suction Boost (Inlet_Pressure plot):

    Slope: Positive.
    Physics: Validated. Higher inlet pressure gives the compressor a "head start," resulting in higher outlet pressure for the same effort.

3. The "Flat" Variables (Feature Selection Confirmation)
Look at Outlet_Flow_Rate and Outlet_Temperature.

Observation: They are perfectly flat.
Interpretation: The model has zeroed out these coefficients (or given them zero importance).
Why this is correct:
The model prefers Outlet_Flow_Rate_SP over Outlet_Flow_Rate. This is good! For a proactive "Digital Twin," you want to predict based on Control Intent (SP), not just sensor correlations. If the flow sensor fails, your model won't break because it relies on the Setpoint.

Final Verdict
We have successfully navigated the "Controller Trap."

We included Setpoints, which split the data into the correct vertical "bands."
We used Ridge/Splines, which captured the linear physics (slopes) within those bands.
We used ICE Plots, which prove that the Flow vs. Pressure relationship (the slope) holds true regardless of which Setpoint band you are in.

Baseline: The model predicts the "Blue Lines" (Healthy State).
Anomaly: When the machine fouls, the actual data points will start falling below the lowest blue band in the ICE plot (or drifting down across bands).
Action: Calculate Residual = Actual - Prediction. When the residual trend hits -3 or -4 sigma (based on your training noise), trigger the alert.

In [24]:
lr_residuals_baseline = calculate_residuals(lr_model, X_baseline, y_baseline)
fig, ax = pyplot.subplots(1, 1)
autocorrelation_plot(lr_residuals_baseline, ax)
pyplot.title("Residual ACF")
fname = os.path.join(image_folder, "linear_model_residuals_acf.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

The ACF plot (bottom image) shows that the autocorrelation is mostly contained within the confidence intervals (the gray dashed lines). This confirms that the residuals are essentially "white noise" (independent and identically distributed). This is the ideal prerequisite for CUSUM, which relies on the assumption of independent errors.

In [25]:
lr_residuals = calculate_residuals(lr_model, X, y)

In [26]:
lr_model_fouling_dates, lr_model_cusum, lr_model_alarm_mask = predict_fouling_onset(
    lr_residuals, baseline_mask, shutin_mask, "neg"
)
fname = os.path.join(image_folder, "linear_model_fouling_summary.png")
plot_fouling_summary(
    y,
    y - lr_residuals,
    baseline_mask,
    shutin_mask,
    lr_model_cusum,
    lr_model_alarm_mask,
    fname,
)

CUSUM held at 0 during maintenance starting 2024-12-01 00:00:00+00:00

False Alaram Rate for clean condition: 0.00%
Fouling number 1 detected: 2024-11-04 00:00:00+00:00
Signal-to-Noise Ratio: 4.59
Peak SNR: 12.05
Fouling number 2 detected: 2025-02-10 00:00:00+00:00
Signal-to-Noise Ratio: 6.46
Peak SNR: 12.05


In [27]:
spline_model = train_model(
    X_baseline, y_baseline, cv_splits_stratified, "spline", n_trials=1000
)

[I 2026-03-21 16:06:56,424] A new study created in memory with name: compressor_outlet_pressure
[I 2026-03-21 16:06:56,439] Trial 0 finished with value: -120.95214877863664 and parameters: {'alpha': 52.467238209873436, 'l1_ratio': 0.08972169220711353, 'fit_intercept': False, 'n_knots': 4, 'knots': 'uniform', 'degree': 3, 'include_bias': True, 'extrapolation': 'constant'}. Best is trial 0 with value: -120.95214877863664.
[I 2026-03-21 16:06:56,464] Trial 1 finished with value: -4.1357131592689305 and parameters: {'alpha': 0.03195653790604932, 'l1_ratio': 0.01869542033302417, 'fit_intercept': True, 'n_knots': 3, 'knots': 'uniform', 'degree': 2, 'include_bias': True, 'extrapolation': 'linear'}. Best is trial 1 with value: -4.1357131592689305.
[I 2026-03-21 16:06:56,489] Trial 2 finished with value: -3.0777091383496504 and parameters: {'alpha': 0.0036601130188662926, 'l1_ratio': 0.2710819958067318, 'fit_intercept': False, 'n_knots': 3, 'knots': 'quantile', 'degree': 3, 'include_bias': True

Best trial:
  Value: -2.5899029042180257
  Params: 
    alpha: 0.0038448942259298345
    l1_ratio: 0.9965627244426309
    fit_intercept: False
    n_knots: 2
    knots: uniform
    degree: 1
    include_bias: True
    extrapolation: constant

Baselibe Training MAE: 2.4520
Baseline Training R^2: 0.9947
Baseline Training adjusted R^2: 0.9944
Baseline Residuals Mean: -0.000
Baseline Residuals Std: 2.808


In [28]:
fname = os.path.join(image_folder, "spline_model_learning_curve.png")
visualize_learning_curve(
    spline_model,
    X_baseline,
    y_baseline,
    np.linspace(0.1, 1.0, 10),
    cv_splits_stratified,
    "neg_mean_absolute_error",
    True,
    True,
    RANDOM_SEED,
    fname,
)

In [29]:
fig, ax = pyplot.subplots(figsize=(25, 15))
PartialDependenceDisplay.from_estimator(
    spline_model,
    X_baseline,
    X_baseline.columns.tolist(),
    kind="both",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    ax=ax,
)
pyplot.tight_layout()
fname = os.path.join(image_folder, "spline_model_partial_dependence.png")
pyplot.savefig(fname, dpi=150, bbox_inches="tight")
pyplot.close(fig)

As for each data shuffle, I get different knot configuration (but degree=1 is almost constant), data seems strictly linear and using spline for such limited data results in unstability. I continue with linear model in production.

### Assumptions

Here are the key assumptions you are making by adopting this "Global Shuffled Baseline" strategy.

It is crucial to document these, as violating any of them in the future could lead to false alarms or missed detections.
1. Physics is Time-Invariant (The "Memoryless" Assumption)

    This is the justification for shuffling your time-series data.

- The Assumption: You assume that the physics of a healthy compressor does not change over time.
- The Implication: A datapoint collected in August (State A) is physically identical to a datapoint collected in December (State A), provided the machine was healthy in both instances. Therefore, the chronological order of training samples does not matter for learning the input/output relationship—only the state matters.
2. Completeness of Operating Regimes (Interpolation vs. Extrapolation)
- The Assumption: You assume that the "Healthy" training period (Aug–Dec) contains all the Setpoints (SP) and Flow regimes that the machine will encounter in the future.
- The Implication: Your model has learned to predict behavior at SP=80, 100, 140 and 180.
If the operators decide to run the machine at SP=110 (between known points), the model will work (Interpolation).
If they decide to run at SP=200 (outside known points), the model will likely fail or give high errors because it is forced to extrapolate into unknown territory.
3. Input Sensor Integrity (Trust in Upstream Data)
- The Assumption: You assume that while the Compressor might foul, the Sensors (Inlet Flow, Inlet Pressure, Temperature) remain perfectly healthy and calibrated.
- The Implication: The model relies on the Inlet Pressure to predict the Outlet Pressure. If the Inlet Pressure sensor starts drifting due to its own electrical fault, your model will produce a wrong prediction, leading to a False Positive alert for compressor fouling.
4. Stationarity of Control Logic
- The Assumption: You assume the PID controller tuning (the logic that drives the machine to meet the Setpoint) has not been changed between the Training period and the Test period.
- The Implication: If a control engineer re-tuned the PID loop in January to make the compressor react faster or slower, the relationship between Setpoint and Actual_Pressure changes. Your model would interpret this new control behavior as an anomaly.
5. Thermodynamic Equilibrium (Steady State)
- The Assumption: You assume the daily samples represent "Steady State" operation, not transient startups or shutdowns.
- The Implication: Since you are using daily batch data (or filtered data), you assume you aren't capturing the exact moment the machine is ramping up from 0 to 100. If a data point captures a transient ramp-up, the pressure will naturally be lagging the setpoint. The model might flag this lag as a performance deficit (fouling) unless those transient points are rigorously filtered out.
6. Linearity of Correction Factors
- The Assumption: By using Ridge Regression (or similar linear models), you assume that the impact of secondary variables (Ambient Humidity, Off-Gas Fraction) is roughly linear or additive.
- The Implication: While fluid dynamics are non-linear, for a proactive maintenance model, we assume that within the standard operating window, the relationship is "linear enough" to set a baseline. We assume we don't need a complex Neural Network to capture extreme edge-case thermodynamics.

Summary for Stakeholders
If you need to present this, you can summarize it as:

"We are assuming the machine's behavior is consistent when healthy, regardless of the date. We are also assuming that we have seen examples of all standard operating speeds. If the machine is pushed to a speed we haven't seen before, or if the inlet sensors fail, the model will require retraining."